<div align="left">

<h2><b>Simple CNN</b></h2>

</div>


#### <u><strong>Setup.</strong></u>

In [1]:
# General Setup (depends on how and where one runs the code)
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/London School of Economics/Deep Learning/DeepLearningProject/Code')

Mounted at /content/drive


In [2]:
# Needed packages
import pandas as pd
import os
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import numpy as np
import requests
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, losses, regularizers
from tensorflow.keras.callbacks import Callback, ReduceLROnPlateau
from PIL import Image, ImageFont
from sklearn.metrics import confusion_matrix
import seaborn as sns
import zipfile

# Setting up directories
PHOTO_DIR  = "../Input/GoogleStreetViewImagesNew.zip"

#### <u><strong>Precursor Functions.</strong></u>

In [3]:
###################
# I. Data loading #
###################
# Sources:
# Assignment I dataloading fn &
# https://medium.com/@kumudtraveldiaries/step-by-step-preprocessing-guide-for-images-in-both-cnn-and-dense-layer-pipelines-1994c3ad3e87

def dataloading(csv_path="../Output/images_bra.csv",
                zip_path=PHOTO_DIR,
                img_size=128,
                batch_size=32,
                preprocess_fn=None,
                variable="income_group"):

    # 1. Load CSV
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=[variable]).reset_index(drop=True)

    # 2. Build group IDs (file_path already references zip contents)
    df["loc_id"] = df["file_path"].apply(
        lambda f: os.path.basename(f).split("_h")[0]
    )

    # 3. Train / Val / Test split at location level
    locs = df["loc_id"].unique()
    np.random.seed(50)
    np.random.shuffle(locs)
    n_test = int(len(locs) * 0.15)
    n_val = int(len(locs) * 0.15)
    test_locs = locs[:n_test]
    val_locs = locs[n_test:n_test+n_val]
    train_locs = locs[n_test+n_val:]
    train_df = df[df["loc_id"].isin(train_locs)]
    val_df   = df[df["loc_id"].isin(val_locs)]
    test_df  = df[df["loc_id"].isin(test_locs)]
    train_labels = train_df[variable].values.astype(np.int64)
    val_labels   = val_df[variable].values.astype(np.int64)
    test_labels  = test_df[variable].values.astype(np.int64)
    splits = {
        "train": (train_df["file_path"].values, train_labels),
        "val":   (val_df["file_path"].values, val_labels),
        "test":  (test_df["file_path"].values, test_labels),
    }
    for k, (p, l) in splits.items():
        print(f"{k}: {len(p)} images")

    # 4. Open zip once and keep in memory
    zip_ref = zipfile.ZipFile(zip_path, "r")

    # 5. Image loading function
    def load_image_from_zip(path, label):
        def _read(path_tensor):
            p = path_tensor.numpy().decode("utf-8")
            img_data = zip_ref.read(p)
            img = tf.image.decode_jpeg(img_data, channels=3)
            img = tf.image.resize(img, [img_size, img_size])
            img = tf.cast(img, tf.float32)
            if preprocess_fn:
                img = preprocess_fn(img)
            else:
                img = img / 255.0
            return img
        img = tf.py_function(_read, [path], tf.float32)
        img.set_shape([img_size, img_size, 3])
        return img, label

    # 6. Build tf.data pipelines
    def make_dataset(file_paths, lbls, is_train=False):
        ds = tf.data.Dataset.from_tensor_slices((file_paths, lbls))
        if is_train:
            ds = ds.shuffle(len(file_paths))
        ds = ds.map(load_image_from_zip, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        return ds

    train_ds = make_dataset(*splits["train"], is_train=True)
    val_ds   = make_dataset(*splits["val"])
    test_ds  = make_dataset(*splits["test"])

    return train_ds, val_ds, test_ds

In [4]:
#####################
# II. Running model #
#####################
def run_model(model,
              train_ds,
              val_ds,
              test_ds,
              variable="income_group",
              n_epochs=50):

    # 1. To export Results
    export_path = f"../Results/SimpleCNN_{variable}.csv"

    # 2. Compile
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=1e-4,
            weight_decay=1e-3
        ),
        loss="sparse_categorical_crossentropy",
        metrics= ['accuracy']
    )

    # 3. Train
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=n_epochs,
        verbose=1,
        callbacks=[
            ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=3,
                min_lr=1e-6,
                verbose=1
            )
        ]
    )

    # 5. Save history
    history_df = pd.DataFrame(history.history)
    history_df["epoch"] = np.arange(1, len(history_df) + 1)
    history_df.to_csv(export_path, index=False)

    return model, history

#### <u><strong>Simple CNN.</strong></u>

In [5]:
train_ds, val_ds, test_ds = dataloading()

train: 15338 images
val: 3284 images
test: 3284 images


In [6]:
# Model
model_simple_nominal = models.Sequential([
    # Input Layer
    layers.Input(shape=(128, 128, 3)),

    # Data augmentation
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.1),

    # Conv Block 0
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Conv Block 1
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Conv Block 2
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # GAP Layer
    layers.GlobalAveragePooling2D(),

    # Output Layer
    layers.Dense(3, activation='softmax')
])

model_simple_nominal.summary()
run_model(model_simple_nominal, train_ds, val_ds, test_ds, variable = "income_group")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           38

 Total params: 437,283 (1.67 MB)

 Trainable params: 436,131 (1.66 MB)

 Non-trainable params: 1,152 (4.50 KB)

Epoch 1/50
480/480 ━━━━━━━━━━━━━━━━━━━━ 92s 144ms/step - accuracy: 0.4289 - loss: 1.0711 - val_accuracy: 0.3758 - val_loss: 1.1036 - learning_rate: 1.0000e-04
Epoch 2/50
480/480 ━━━━━━━━━━━━━━━━━━━━ 45s 93ms/step - accuracy: 0.4746 - loss: 1.0221 - val_accuracy: 0.4202 - val_loss: 1.1550 - learning_rate: 1.0000e-04
Epoch 3/50
480/480 ━━━━━━━━━━━━━━━━━━━━ 45s 94ms/step - accuracy: 0.4995 - loss: 0.9971 - val_accuracy: 0.4686 - val_loss: 1.0633 - learning_rate: 1.0000e-04
Epoch 4/50
480/480 ━━━━━━━━━━━━━━━━━━━━ 45s 94ms/step - accuracy: 0.5159 - loss: 0.9781 - val_accuracy: 0.4278 - val_loss: 1.1700 - learning_rate: 1.0000e-04
Epoch 5/50
480/480 ━━━━━━━━━━━━━━━━━━━━ 45s 94ms/step - accuracy: 0.5381 - loss: 0.9600 - val_accuracy: 0.4699 - val_loss: 1.0507 - learning_rate: 1.0000e-04
Epoch 6/50
480/480 ━━━━━━━━━━━━━━━━━━━━ 45s 94ms/step - accuracy: 0.5436 - loss: 0.9452 - val_accuracy: 0.4543 - val_loss: 1.1269 - learning_rate: 1.0000e-04
Epoch 7/50
480/480 ━━━━━━━━━━━━━━━━━━━━ 45s 94ms/st

(<Sequential name=sequential, built=True>,
 <keras.src.callbacks.history.History at 0x7f368784fe30>)